In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
data = pd.read_csv('Data/Churn_Modelling.csv')
dataset = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [3]:
column_order = ['Geography', 'Gender','EstimatedSalary', 'CreditScore', 'Balance', 'Age', 'Tenure', 
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'Exited']
dataset = dataset[column_order]
features = dataset.drop('Exited', axis=1)

In [4]:
features.head(2)

,Geography,Gender,EstimatedSalary,CreditScore,Balance,Age,Tenure,NumOfProducts,HasCrCard,IsActiveMember
0,France,Female,101348.88,619,0.00,42,2,1,1,1
1,Spain,Female,112542.58,608,83807.86,41,1,1,0,1


In [5]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
features.iloc[:,1] = le.fit_transform(features.iloc[:,1])

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(drop='first'), ['Geography'])], remainder='passthrough')
features = ct.fit_transform(features)

In [7]:
features_dataset = pd.DataFrame(features, columns=ct.get_feature_names_out())
features_dataset.head(2)

,encoder__Geography_Germany,encoder__Geography_Spain,remainder__Gender,remainder__EstimatedSalary,remainder__CreditScore,remainder__Balance,remainder__Age,remainder__Tenure,remainder__NumOfProducts,remainder__HasCrCard,remainder__IsActiveMember
0,0.0,0.0,0,101348.88,619,0.0,42,2,1,1,1
1,0.0,1.0,0,112542.58,608,83807.86,41,1,1,0,1


In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
features_dataset = sc.fit_transform(features_dataset)

In [20]:
x = features_dataset
y = data.iloc[:,-1].values

In [21]:
nb_ones = int(np.sum(y))
counter = 0
indix_to_remove = []

for i in range(y.shape[0]):
    if y[i] == 0 :
        counter += 1
        if counter > nb_ones :
            indix_to_remove.append(i)
x = np.delete(x, indix_to_remove, axis=0)
y = np.delete(y, indix_to_remove, axis=0)

In [22]:
shuffle_indix = np.arange(y.shape[0])
np.random.shuffle(shuffle_indix)

x = x[shuffle_indix]
y = y[shuffle_indix]

In [23]:
samples_counts = x.shape[0]

train_count = int(0.8 * samples_counts)
validation_count = int(0.1 * samples_counts)
test_count = int(0.1 * samples_counts)

x_train = x[:train_count]
y_train = y[:train_count]

x_valid = x[train_count:train_count+validation_count]
y_valid = y[train_count:train_count+validation_count]

x_test = x[train_count+validation_count:]
y_test = y[train_count+validation_count:]

In [24]:
np.savez('data_train', inputs=x_train, targets=y_train)
np.savez('data_validation', inputs=x_valid, targets=y_valid)
np.savez('data_test', inputs=x_test, targets=y_test)